In [2]:
# ============================================================================
# SECTION 1: IMPORT REQUIRED LIBRARIES
# ============================================================================

import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Spark imports for gold-layer data access
from pyspark.sql import SparkSession
from pyspark import SparkConf




In [3]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "http://localhost:9878")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "tazama")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "tazama")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
metrics_warehouse = f"{WAREHOUSE_ROOT}/metrics"

# Source paths for gold-layer metrics
tms_metrics_path = f"{metrics_warehouse}/transaction_monitoring"
alert_metrics_path = f"{metrics_warehouse}/alerts"
case_metrics_path = f"{metrics_warehouse}/cases"
fraud_metrics_path = f"{metrics_warehouse}/fraud"

print(f"\n✓ Warehouse paths configured:")
print(f"  Metrics warehouse: {metrics_warehouse}")

# Load pre-aggregated metrics from gold layer
try:
    print("\n[Loading Gold-Layer Metrics...]")
    
    # Load TMS metrics
    tms_metrics = spark.read.parquet(tms_metrics_path)
    print("  ✓ TMS metrics loaded")
    
    # Load Alert metrics
    alert_metrics = spark.read.parquet(alert_metrics_path)
    print("  ✓ Alert metrics loaded")
    
    # Load Case metrics
    case_metrics = spark.read.parquet(case_metrics_path)
    print("  ✓ Case metrics loaded")
    
    # Load Fraud metrics
    fraud_metrics = spark.read.parquet(fraud_metrics_path)
    print("  ✓ Fraud metrics loaded")
    
    print("\n✓ All gold-layer metrics loaded successfully!")
    
except Exception as e:
    print(f"✗ Error loading metrics: {e}")
    print("  Ensure metrics have been pre-aggregated in gold layer (user_story_asim.ipynb Section 5)")


26/04/17 18:41:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark Version: 3.4.2

✓ Warehouse paths configured:
  Metrics warehouse: /opt/Tazama_Hudi_warehouse/metrics

[Loading Gold-Layer Metrics...]
✗ Error loading metrics: [PATH_NOT_FOUND] Path does not exist: file:/opt/Tazama_Hudi_warehouse/metrics/transaction_monitoring.
  Ensure metrics have been pre-aggregated in gold layer (user_story_asim.ipynb Section 5)


In [4]:
# ============================================================================
# SECTION 3: DEFINE DATE RANGE & GRANULARITY PARAMETERS
# ============================================================================

from pyspark.sql import functions as F

# ============================================================================
# 3.1: DATE RANGE SELECTION
# ============================================================================

# PARAMETER: Select date range type
# Options: 'custom', 'last_7_days', 'last_30_days', 'last_90_days', 'year', 'all_history'
date_range_type = 'last_30_days'

# PARAMETER: Custom date range (used when date_range_type='custom')
custom_start_date = '2024-01-01'
custom_end_date = '2024-01-31'

# Function to calculate date range
def get_date_range(range_type, custom_start=None, custom_end=None):
    """Calculate date range based on selection"""
    end_date = datetime.now().date()
    
    if range_type == 'custom':
        start_date = datetime.strptime(custom_start, '%Y-%m-%d').date()
        end_date = datetime.strptime(custom_end, '%Y-%m-%d').date()
    elif range_type == 'last_7_days':
        start_date = end_date - timedelta(days=7)
    elif range_type == 'last_30_days':
        start_date = end_date - timedelta(days=30)
    elif range_type == 'last_90_days':
        start_date = end_date - timedelta(days=90)
    elif range_type == 'year':
        start_date = end_date - timedelta(days=365)
    elif range_type == 'all_history':
        start_date = datetime.strptime('2020-01-01', '%Y-%m-%d').date()
    else:
        start_date = end_date - timedelta(days=30)
    
    return start_date, end_date

# Get active date range
dashboard_start_date, dashboard_end_date = get_date_range(
    date_range_type, 
    custom_start_date, 
    custom_end_date
)

print("="*80)
print("EXECUTIVE OVERVIEW DASHBOARD - PARAMETERS")
print("="*80)
print(f"DATE RANGE SELECTION:")
print(f"   Period: {date_range_type}")
print(f"   Start Date: {dashboard_start_date}")
print(f"   End Date: {dashboard_end_date}")
print(f"   Days in Range: {(dashboard_end_date - dashboard_start_date).days + 1}")

# ============================================================================
# 3.2: TIME GRANULARITY SELECTION
# ============================================================================

# PARAMETER: Select time granularity
# Options: 'daily', 'weekly', 'monthly', 'quarterly'
time_granularity = 'monthly'

print(f"TIME GRANULARITY: {time_granularity.upper()}")

# Function to aggregate data to selected granularity
def aggregate_to_granularity(df, metric_date_col, granularity):
    """Aggregate metrics to selected time granularity"""
    # Determine the appropriate date column
    date_col = metric_date_col if metric_date_col in df.columns else "metric_month"
    
    if granularity == 'daily':
        return df.withColumn("period", F.col(date_col))
    elif granularity == 'weekly':
        return df.withColumn(
            "period", 
            F.date_format(
                F.date_sub(F.to_date(F.col(date_col)), F.dayofweek(F.to_date(F.col(date_col)))-1),
                "yyyy-MM-dd"
            )
        )
    elif granularity == 'monthly':
        return df.withColumn(
            "period",
            F.date_format(F.to_date(F.col(date_col)), "yyyy-MM")
        )
    elif granularity == 'quarterly':
        return df.withColumn(
            "period",
            F.concat(
                F.year(F.to_date(F.col(date_col))),
                F.lit("-Q"),
                F.quarter(F.to_date(F.col(date_col)))
            )
        )
    else:
        return df.withColumn("period", F.col(date_col))

# ============================================================================
# 3.3: FILTER METRICS TO DATE RANGE
# ============================================================================

start_str = dashboard_start_date.strftime('%Y-%m-%d')
end_str = dashboard_end_date.strftime('%Y-%m-%d')

print(f"Data Schema Check:")
print(f"   TMS columns: {tms_metrics.columns}")
print(f"   Alert columns: {alert_metrics.columns}")
print(f"   Case columns: {case_metrics.columns}")
print(f"   Fraud columns: {fraud_metrics.columns}")

# Use metric_month for filtering since parquet files use that column
# For metrics that have metric_date, use that; otherwise use metric_month
if "metric_date" in fraud_metrics.columns:
    fraud_metrics_filtered = fraud_metrics.filter(
        (F.col("metric_date") >= start_str) & (F.col("metric_date") <= end_str)
    )
else:
    fraud_metrics_filtered = fraud_metrics

if "metric_date" in alert_metrics.columns:
    alert_metrics_filtered = alert_metrics.filter(
        (F.col("metric_date") >= start_str) & (F.col("metric_date") <= end_str)
    )
else:
    alert_metrics_filtered = alert_metrics

if "metric_date" in case_metrics.columns:
    case_metrics_filtered = case_metrics.filter(
        (F.col("metric_date") >= start_str) & (F.col("metric_date") <= end_str)
    )
else:
    case_metrics_filtered = case_metrics

if "metric_date" in tms_metrics.columns:
    tms_metrics_filtered = tms_metrics.filter(
        (F.col("metric_date") >= start_str) & (F.col("metric_date") <= end_str)
    )
else:
    tms_metrics_filtered = tms_metrics

print(f"\n✓ Metrics loaded and ready for analysis:")
print(f"   TMS Records: {tms_metrics_filtered.count()}")
print(f"   Alert Records: {alert_metrics_filtered.count()}")
print(f"   Case Records: {case_metrics_filtered.count()}")
print(f"   Fraud Records: {fraud_metrics_filtered.count()}")


EXECUTIVE OVERVIEW DASHBOARD - PARAMETERS
DATE RANGE SELECTION:
   Period: last_30_days
   Start Date: 2026-03-18
   End Date: 2026-04-17
   Days in Range: 31
TIME GRANULARITY: MONTHLY
Data Schema Check:


NameError: name 'tms_metrics' is not defined

In [10]:
# ============================================================================
# SECTION 4: CALCULATE FRAUD PERFORMANCE METRICS
# ============================================================================

print("\n" + "="*80)
print("FRAUD PERFORMANCE METRICS CALCULATION")
print("="*80)

# Aggregate to selected granularity
fraud_metrics_agg = aggregate_to_granularity(
    fraud_metrics_filtered, "metric_date", time_granularity
).groupBy("period").agg(
    F.sum("confirmed_cases").alias("confirmed_cases"),
    F.sum("true_positives").alias("true_positives"),
    F.sum("false_positives").alias("false_positives"),
    F.sum("false_negatives").alias("false_negatives"),
    F.sum("transactions_evaluated").alias("transactions_evaluated"),
    F.avg("precision").alias("precision"),
    F.avg("recall_tpr").alias("recall_tpr")
).orderBy("period")

# Convert to Pandas for calculation
fraud_perf_pd = fraud_metrics_agg.toPandas()

# METRIC 1: Fraud Rate = Confirmed fraud / All transactions evaluated
fraud_perf_pd['fraud_rate_pct'] = (
    fraud_perf_pd['confirmed_cases'] / fraud_perf_pd['transactions_evaluated'] * 100
).round(4)

# METRIC 2: Precision = TP / (TP + FP)
fraud_perf_pd['precision_calc'] = (
    fraud_perf_pd['true_positives'] / 
    (fraud_perf_pd['true_positives'] + fraud_perf_pd['false_positives'])
).round(4) * 100

# METRIC 3: True Positive Rate (TPR) = TP / (TP + FN)
fraud_perf_pd['tpr_calc'] = (
    fraud_perf_pd['true_positives'] / 
    (fraud_perf_pd['true_positives'] + fraud_perf_pd['false_negatives'])
).round(4) * 100

# Replace inf and nan with 0
fraud_perf_pd = fraud_perf_pd.fillna(0).replace([np.inf, -np.inf], 0)

print("\n✓ Fraud Performance Metrics Calculated:")
print(f"   1. Fraud Rate (Confirmed fraud / All transactions)")
print(f"   2. Precision (TP / (TP + FP))")
print(f"   3. True Positive Rate / Recall (TP / (TP + FN))")

# Calculate current period metrics
current_fraud_rate = fraud_perf_pd['fraud_rate_pct'].iloc[-1] if len(fraud_perf_pd) > 0 else 0
current_precision = fraud_perf_pd['precision_calc'].iloc[-1] if len(fraud_perf_pd) > 0 else 0
current_tpr = fraud_perf_pd['tpr_calc'].iloc[-1] if len(fraud_perf_pd) > 0 else 0

print(f"Current Period Values:")
print(f"   Fraud Rate: {current_fraud_rate:.4f}%")
print(f"   Precision: {current_precision:.4f}%")
print(f"   TPR (Recall): {current_tpr:.4f}%")

fraud_perf_pd_display = fraud_perf_pd[['period', 'fraud_rate_pct', 'precision_calc', 'tpr_calc']].head()
print(f"\nSample Data:")
print(fraud_perf_pd_display.to_string(index=False))



FRAUD PERFORMANCE METRICS CALCULATION

✓ Fraud Performance Metrics Calculated:
   1. Fraud Rate (Confirmed fraud / All transactions)
   2. Precision (TP / (TP + FP))
   3. True Positive Rate / Recall (TP / (TP + FN))
Current Period Values:
   Fraud Rate: 0.0000%
   Precision: 0.0000%
   TPR (Recall): 0.0000%

Sample Data:
Empty DataFrame
Columns: [period, fraud_rate_pct, precision_calc, tpr_calc]
Index: []


In [11]:
# ============================================================================
# SECTION 5: CALCULATE OPERATIONAL EFFICIENCY METRICS
# ============================================================================

print("\n" + "="*80)
print("OPERATIONAL EFFICIENCY METRICS CALCULATION")
print("="*80)

# Aggregate alert metrics to selected granularity
alert_metrics_agg = aggregate_to_granularity(
    alert_metrics_filtered, "metric_date", time_granularity
).groupBy("period").agg(
    F.sum("transactions_generated_alert").alias("transactions_generated_alert"),
    F.sum("transactions_evaluated").alias("transactions_evaluated"),
    F.sum("overall_interdiction_rate_pct").alias("overall_interdiction_rate_sum"),
    F.avg("evaluated_alert_rate_pct").alias("evaluated_alert_rate_pct")
).orderBy("period")

# Aggregate case metrics for MTTD
case_metrics_agg = aggregate_to_granularity(
    case_metrics_filtered, "metric_date", time_granularity
).groupBy("period").agg(
    F.sum("total_cases_created").alias("total_cases_created"),
    F.sum("refuted_cases").alias("refuted_cases"),
    F.avg("avg_mttd_days").alias("avg_mttd_days")
).orderBy("period")

# Convert to Pandas
alert_perf_pd = alert_metrics_agg.toPandas()
case_perf_pd = case_metrics_agg.toPandas()

# METRIC 1: False Positive Rate (FPR) = Refuted cases / Transactions with alerts
case_perf_pd['false_positive_rate_pct'] = (
    case_perf_pd['refuted_cases'] / case_perf_pd['total_cases_created'] * 100
).round(2)

# METRIC 2: Evaluated Alert Rate = Transactions with alerts / Transactions evaluated
alert_perf_pd['evaluated_alert_rate_calc'] = alert_perf_pd['evaluated_alert_rate_pct'].round(2)

# METRIC 3: Overall Interdiction Rate (average)
alert_perf_pd['overall_interdiction_rate_pct'] = (
    alert_perf_pd['overall_interdiction_rate_sum'] / 
    alert_perf_pd.shape[0]
).round(2)

# METRIC 4: Mean Time to Disposition (MTTD) - already calculated
case_perf_pd['mttd_days'] = case_perf_pd['avg_mttd_days'].round(2)

# Replace inf and nan with 0
alert_perf_pd = alert_perf_pd.fillna(0).replace([np.inf, -np.inf], 0)
case_perf_pd = case_perf_pd.fillna(0).replace([np.inf, -np.inf], 0)

print("\n✓ Operational Efficiency Metrics Calculated:")
print(f"   1. False Positive Rate (Refuted cases / Alert transactions)")
print(f"   2. Evaluated Alert Rate (Alert transactions / Evaluated transactions)")
print(f"   3. Overall Interdiction Rate (Interdicted / Evaluated)")
print(f"   4. Mean Time to Disposition (Days)")

# Calculate current period metrics
current_fpr = case_perf_pd['false_positive_rate_pct'].iloc[-1] if len(case_perf_pd) > 0 else 0
current_evaluated_alert_rate = alert_perf_pd['evaluated_alert_rate_calc'].iloc[-1] if len(alert_perf_pd) > 0 else 0
current_interdiction_rate = alert_perf_pd['overall_interdiction_rate_pct'].iloc[-1] if len(alert_perf_pd) > 0 else 0
current_mttd = case_perf_pd['mttd_days'].iloc[-1] if len(case_perf_pd) > 0 else 0

print(f"\n📊 Current Period Values:")
print(f"   False Positive Rate: {current_fpr:.2f}%")
print(f"   Evaluated Alert Rate: {current_evaluated_alert_rate:.2f}%")
print(f"   Interdiction Rate: {current_interdiction_rate:.2f}%")
print(f"   MTTD: {current_mttd:.2f} days")

print(f"\nAlert Metrics Sample:")
print(alert_perf_pd[['period', 'evaluated_alert_rate_calc', 'overall_interdiction_rate_pct']].head().to_string(index=False))

print(f"\nCase Metrics Sample:")
print(case_perf_pd[['period', 'false_positive_rate_pct', 'mttd_days']].head().to_string(index=False))



OPERATIONAL EFFICIENCY METRICS CALCULATION

✓ Operational Efficiency Metrics Calculated:
   1. False Positive Rate (Refuted cases / Alert transactions)
   2. Evaluated Alert Rate (Alert transactions / Evaluated transactions)
   3. Overall Interdiction Rate (Interdicted / Evaluated)
   4. Mean Time to Disposition (Days)

📊 Current Period Values:
   False Positive Rate: 0.00%
   Evaluated Alert Rate: 0.00%
   Interdiction Rate: 0.00%
   MTTD: 0.00 days

Alert Metrics Sample:
 period  evaluated_alert_rate_calc  overall_interdiction_rate_pct
2026-04                        0.0                            0.0

Case Metrics Sample:
 period  false_positive_rate_pct  mttd_days
2026-04                      0.0        0.0


In [12]:
# ============================================================================
# SECTION 6: CALCULATE FINANCIAL IMPACT METRICS
# ============================================================================

print("\n" + "="*80)
print("FINANCIAL IMPACT METRICS CALCULATION")
print("="*80)

# Aggregate fraud metrics for financial calculations
fraud_metrics_financial = aggregate_to_granularity(
    fraud_metrics_filtered, "metric_date", time_granularity
).groupBy("period").agg(
    F.sum("confirmed_cases").alias("confirmed_cases"),
    F.sum("confirmed_fraud_value").alias("confirmed_fraud_value"),
    F.sum("prevented_fraud_value").alias("prevented_fraud_value")
).orderBy("period")

# Convert to Pandas
financial_pd = fraud_metrics_financial.toPandas()

# METRIC 1: Total Fraud Value = Sum of confirmed fraud amounts
financial_pd['total_fraud_value'] = financial_pd['confirmed_fraud_value'].round(2)

# METRIC 2: Prevented Fraud Value = Sum of interdicted confirmed fraud amounts
financial_pd['prevented_fraud_value_calc'] = financial_pd['prevented_fraud_value'].round(2)

# METRIC 3: Average Fraud Value = Total fraud value / number of confirmed cases
financial_pd['average_fraud_value'] = (
    financial_pd['confirmed_fraud_value'] / financial_pd['confirmed_cases']
).round(2)

# Replace inf and nan with 0
financial_pd = financial_pd.fillna(0).replace([np.inf, -np.inf], 0)

print("\n✓ Financial Impact Metrics Calculated:")
print(f"   1. Total Fraud Value (Confirmed fraud losses)")
print(f"   2. Prevented Fraud Value (Value of interdicted fraud)")
print(f"   3. Average Fraud Value (Per confirmed case)")

# Calculate current period metrics
current_total_fraud_value = financial_pd['total_fraud_value'].iloc[-1] if len(financial_pd) > 0 else 0
current_prevented_value = financial_pd['prevented_fraud_value_calc'].iloc[-1] if len(financial_pd) > 0 else 0
current_avg_fraud_value = financial_pd['average_fraud_value'].iloc[-1] if len(financial_pd) > 0 else 0

print(f"\n💰 Current Period Values:")
print(f"   Total Fraud Value: ${current_total_fraud_value:,.2f}")
print(f"   Prevented Fraud Value: ${current_prevented_value:,.2f}")
print(f"   Average Fraud Value: ${current_avg_fraud_value:,.2f}")

print(f"\nFinancial Metrics Sample:")
print(financial_pd[['period', 'total_fraud_value', 'prevented_fraud_value_calc', 'average_fraud_value']].head().to_string(index=False))



FINANCIAL IMPACT METRICS CALCULATION

✓ Financial Impact Metrics Calculated:
   1. Total Fraud Value (Confirmed fraud losses)
   2. Prevented Fraud Value (Value of interdicted fraud)
   3. Average Fraud Value (Per confirmed case)

💰 Current Period Values:
   Total Fraud Value: $0.00
   Prevented Fraud Value: $0.00
   Average Fraud Value: $0.00

Financial Metrics Sample:
Empty DataFrame
Columns: [period, total_fraud_value, prevented_fraud_value_calc, average_fraud_value]
Index: []


In [13]:
# ============================================================================
# SECTION 7: COMPUTE TREND INDICATORS (DIRECTION & % CHANGE)
# ============================================================================

print("\n" + "="*80)
print("TREND INDICATORS CALCULATION")
print("="*80)

def calculate_trend_direction(current_value, previous_value, higher_is_better=True):
    """Calculate trend direction (↑ improving, → stable, ↓ declining)"""
    if previous_value == 0 or pd.isna(previous_value):
        return "→"
    
    if current_value > previous_value:
        return "↑" if higher_is_better else "↓"
    elif current_value < previous_value:
        return "↓" if higher_is_better else "↑"
    else:
        return "→"

def calculate_pct_change(current_value, previous_value):
    """Calculate percentage change"""
    if previous_value == 0 or pd.isna(previous_value):
        return 0
    return ((current_value - previous_value) / abs(previous_value)) * 100

# Combine all metrics for trend calculation
trends_data = {
    'period': fraud_perf_pd['period'],
    'fraud_rate': fraud_perf_pd['fraud_rate_pct'],
    'precision': fraud_perf_pd['precision_calc'],
    'tpr': fraud_perf_pd['tpr_calc'],
    'fpr': case_perf_pd['false_positive_rate_pct'],
    'evaluated_alert_rate': alert_perf_pd['evaluated_alert_rate_calc'],
    'interdiction_rate': alert_perf_pd['overall_interdiction_rate_pct'],
    'mttd': case_perf_pd['mttd_days'],
    'total_fraud_value': financial_pd['total_fraud_value'],
    'prevented_value': financial_pd['prevented_fraud_value_calc'],
}

trends_df = pd.DataFrame(trends_data)

# Calculate trend indicators for each metric
trends_df['fraud_rate_change_pct'] = trends_df['fraud_rate'].pct_change() * 100
trends_df['precision_change_pct'] = trends_df['precision'].pct_change() * 100
trends_df['tpr_change_pct'] = trends_df['tpr'].pct_change() * 100
trends_df['fpr_change_pct'] = trends_df['fpr'].pct_change() * 100
trends_df['ear_change_pct'] = trends_df['evaluated_alert_rate'].pct_change() * 100
trends_df['idr_change_pct'] = trends_df['interdiction_rate'].pct_change() * 100
trends_df['mttd_change_pct'] = trends_df['mttd'].pct_change() * 100
trends_df['fraud_value_change_pct'] = trends_df['total_fraud_value'].pct_change() * 100

# Calculate direction indicators
trends_df['fraud_rate_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['fraud_rate'], 
        trends_df['fraud_rate'].iloc[-2] if len(trends_df) >= 2 else row['fraud_rate'],
        higher_is_better=False  # Lower fraud rate is better
    ), axis=1
)

trends_df['precision_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['precision'], 
        trends_df['precision'].iloc[-2] if len(trends_df) >= 2 else row['precision'],
        higher_is_better=True  # Higher precision is better
    ), axis=1
)

trends_df['tpr_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['tpr'], 
        trends_df['tpr'].iloc[-2] if len(trends_df) >= 2 else row['tpr'],
        higher_is_better=True
    ), axis=1
)

trends_df['fpr_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['fpr'], 
        trends_df['fpr'].iloc[-2] if len(trends_df) >= 2 else row['fpr'],
        higher_is_better=False
    ), axis=1
)

trends_df['idr_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['interdiction_rate'], 
        trends_df['interdiction_rate'].iloc[-2] if len(trends_df) >= 2 else row['interdiction_rate'],
        higher_is_better=True
    ), axis=1
)

trends_df['fraud_value_direction'] = trends_df.apply(
    lambda row: calculate_trend_direction(
        row['total_fraud_value'], 
        trends_df['total_fraud_value'].iloc[-2] if len(trends_df) >= 2 else row['total_fraud_value'],
        higher_is_better=False
    ), axis=1
)

# Fill na with 0
trends_df = trends_df.fillna(0).replace([np.inf, -np.inf], 0)

print("\n✓ Trend Indicators Calculated (Direction & % Change)")
print(f"   Metrics tracked: 10 KPIs with period-over-period comparison")
print(f"   Direction symbols: ↑ (improving), → (stable), ↓ (declining)")

# Display current trends
if len(trends_df) > 0:
    latest = trends_df.iloc[-1]
    print(f"\n📊 Latest Trend Summary:")
    print(f"   Fraud Rate: {latest['fraud_rate_direction']} ({latest['fraud_rate_change_pct']:+.1f}%)")
    print(f"   Precision: {latest['precision_direction']} ({latest['precision_change_pct']:+.1f}%)")
    print(f"   TPR: {latest['tpr_direction']} ({latest['tpr_change_pct']:+.1f}%)")
    print(f"   FPR: {latest['fpr_direction']} ({latest['fpr_change_pct']:+.1f}%)")
    print(f"   Interdiction Rate: {latest['idr_direction']} ({latest['idr_change_pct']:+.1f}%)")
    print(f"   Fraud Value: {latest['fraud_value_direction']} ({latest['fraud_value_change_pct']:+.1f}%)")

print(f"\nTrends Sample (Last 5 periods):")
trend_display = trends_df[['period', 'fraud_rate', 'fraud_rate_change_pct', 'fraud_rate_direction',
                           'precision', 'precision_change_pct', 'precision_direction']].tail()
print(trend_display.to_string(index=False))



TREND INDICATORS CALCULATION

✓ Trend Indicators Calculated (Direction & % Change)
   Metrics tracked: 10 KPIs with period-over-period comparison
   Direction symbols: ↑ (improving), → (stable), ↓ (declining)

📊 Latest Trend Summary:
   Fraud Rate: → (+0.0%)
   Precision: → (+0.0%)
   TPR: → (+0.0%)
   FPR: → (+0.0%)
   Interdiction Rate: → (+0.0%)
   Fraud Value: → (+0.0%)

Trends Sample (Last 5 periods):
 period  fraud_rate  fraud_rate_change_pct fraud_rate_direction  precision  precision_change_pct precision_direction
      0         0.0                    0.0                    →        0.0                   0.0                   →


In [14]:
# ============================================================================
# SECTION 8.1: RENDER KPI CARDS (Executive Key Performance Indicators)
# ============================================================================

print("\n" + "="*80)
print("VISUALIZATION: KPI CARDS")
print("="*80)

# Create a dashboard with KPI cards (3x4 grid)
fig_kpis = go.Figure()

# Define KPIs with current values, thresholds, and status
kpis = [
    {
        'title': 'Fraud Rate (%)',
        'value': current_fraud_rate,
        'unit': '%',
        'threshold_good': 2.0,
        'threshold_fair': 5.0,
        'higher_is_better': False
    },
    {
        'title': 'Precision (%)',
        'value': current_precision,
        'unit': '%',
        'threshold_good': 80.0,
        'threshold_fair': 60.0,
        'higher_is_better': True
    },
    {
        'title': 'True Positive Rate (%)',
        'value': current_tpr,
        'unit': '%',
        'threshold_good': 75.0,
        'threshold_fair': 50.0,
        'higher_is_better': True
    },
    {
        'title': 'False Positive Rate (%)',
        'value': current_fpr,
        'unit': '%',
        'threshold_good': 3.0,
        'threshold_fair': 8.0,
        'higher_is_better': False
    },
    {
        'title': 'Evaluated Alert Rate (%)',
        'value': current_evaluated_alert_rate,
        'unit': '%',
        'threshold_good': 15.0,
        'threshold_fair': 25.0,
        'higher_is_better': True
    },
    {
        'title': 'Interdiction Rate (%)',
        'value': current_interdiction_rate,
        'unit': '%',
        'threshold_good': 85.0,
        'threshold_fair': 70.0,
        'higher_is_better': True
    },
    {
        'title': 'MTTD (Days)',
        'value': current_mttd,
        'unit': 'days',
        'threshold_good': 5.0,
        'threshold_fair': 10.0,
        'higher_is_better': False
    },
    {
        'title': 'Total Fraud Value',
        'value': current_total_fraud_value,
        'unit': '$',
        'threshold_good': 1000000,
        'threshold_fair': 5000000,
        'higher_is_better': False
    },
    {
        'title': 'Prevented Fraud Value',
        'value': current_prevented_value,
        'unit': '$',
        'threshold_good': 500000,
        'threshold_fair': 100000,
        'higher_is_better': True
    },
    {
        'title': 'Average Fraud Value',
        'value': current_avg_fraud_value,
        'unit': '$',
        'threshold_good': 5000,
        'threshold_fair': 15000,
        'higher_is_better': False
    },
]

# Create KPI indicators using plotly indicators
fig_kpis = make_subplots(
    rows=4, cols=3,
    specs=[[{'type': 'indicator'}] * 3] * 4,
    vertical_spacing=0.15,
    horizontal_spacing=0.15
)

for idx, kpi in enumerate(kpis):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    
    # Determine status color
    if kpi['higher_is_better']:
        if kpi['value'] >= kpi['threshold_good']:
            color = '#2ecc71'  # Green
        elif kpi['value'] >= kpi['threshold_fair']:
            color = '#f39c12'  # Orange
        else:
            color = '#e74c3c'  # Red
    else:
        if kpi['value'] <= kpi['threshold_good']:
            color = '#2ecc71'  # Green
        elif kpi['value'] <= kpi['threshold_fair']:
            color = '#f39c12'  # Orange
        else:
            color = '#e74c3c'  # Red
    
    fig_kpis.add_trace(
        go.Indicator(
            mode="number+gauge",
            value=kpi['value'],
            title={'text': kpi['title']},
            gauge={
                'axis': {'range': [0, kpi['threshold_fair'] * 1.5]},
                'bar': {'color': color},
                'steps': [
                    {'range': [0, kpi['threshold_good']], 'color': "#e8f5e9"},
                    {'range': [kpi['threshold_good'], kpi['threshold_fair']], 'color': "#fff3cd"},
                    {'range': [kpi['threshold_fair'], kpi['threshold_fair'] * 1.5], 'color': "#f8d7da"},
                ],
            },
            number={'suffix': kpi['unit']},
        ),
        row=row, col=col
    )

fig_kpis.update_layout(
    title_text=f"<b>Executive KPI Dashboard</b><br><sub>Time Period: {time_granularity.upper()} | Date Range: {dashboard_start_date.strftime('%Y-%m-%d')} to {dashboard_end_date.strftime('%Y-%m-%d')}</sub>",
    showlegend=False,
    height=1200,
    width=1400
)

print("✓ KPI Cards visualization created (4x3 grid with 10 metrics)")
print("  Status indicators: 🟢 Good | 🟡 Fair | 🔴 Poor")

# Display the figure
fig_kpis.show()



VISUALIZATION: KPI CARDS
✓ KPI Cards visualization created (4x3 grid with 10 metrics)
  Status indicators: 🟢 Good | 🟡 Fair | 🔴 Poor


In [15]:
# ============================================================================
# SECTION 8.2: RENDER TIME-SERIES CHARTS (Trends Over Time)
# ============================================================================

print("\n" + "="*80)
print("VISUALIZATION: TIME-SERIES TRENDS")
print("="*80)

# Create 3 time-series subplots
fig_trends = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "Fraud Detection Performance Metrics",
        "Alert Processing & Interdiction Metrics",
        "Financial Impact Metrics"
    ),
    specs=[
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": True}]
    ],
    vertical_spacing=0.12
)

# CHART 1: Fraud Performance Metrics
fig_trends.add_trace(
    go.Scatter(
        x=fraud_perf_pd['period'],
        y=fraud_perf_pd['fraud_rate_pct'],
        name='Fraud Rate (%)',
        mode='lines+markers',
        line=dict(color='#e74c3c', width=2),
        marker=dict(size=6)
    ),
    row=1, col=1
)

fig_trends.add_trace(
    go.Scatter(
        x=fraud_perf_pd['period'],
        y=fraud_perf_pd['precision_calc'],
        name='Precision (%)',
        mode='lines+markers',
        line=dict(color='#3498db', width=2),
        marker=dict(size=6)
    ),
    row=1, col=1
)

fig_trends.add_trace(
    go.Scatter(
        x=fraud_perf_pd['period'],
        y=fraud_perf_pd['tpr_calc'],
        name='TPR/Recall (%)',
        mode='lines+markers',
        line=dict(color='#2ecc71', width=2),
        marker=dict(size=6)
    ),
    row=1, col=1
)

# CHART 2: Alert Processing Metrics
fig_trends.add_trace(
    go.Scatter(
        x=alert_perf_pd['period'],
        y=alert_perf_pd['evaluated_alert_rate_calc'],
        name='Evaluated Alert Rate (%)',
        mode='lines+markers',
        line=dict(color='#f39c12', width=2),
        marker=dict(size=6)
    ),
    row=2, col=1
)

fig_trends.add_trace(
    go.Scatter(
        x=alert_perf_pd['period'],
        y=alert_perf_pd['overall_interdiction_rate_pct'],
        name='Interdiction Rate (%)',
        mode='lines+markers',
        line=dict(color='#9b59b6', width=2),
        marker=dict(size=6)
    ),
    row=2, col=1
)

fig_trends.add_trace(
    go.Scatter(
        x=case_perf_pd['period'],
        y=case_perf_pd['mttd_days'],
        name='MTTD (Days)',
        mode='lines+markers',
        line=dict(color='#1abc9c', width=2),
        marker=dict(size=6)
    ),
    row=2, col=1
)

# CHART 3: Financial Metrics
fig_trends.add_trace(
    go.Scatter(
        x=financial_pd['period'],
        y=financial_pd['total_fraud_value'],
        name='Total Fraud Value',
        mode='lines+markers',
        line=dict(color='#c0392b', width=2),
        marker=dict(size=6)
    ),
    row=3, col=1, secondary_y=False
)

fig_trends.add_trace(
    go.Scatter(
        x=financial_pd['period'],
        y=financial_pd['prevented_fraud_value_calc'],
        name='Prevented Fraud Value',
        mode='lines+markers',
        line=dict(color='#27ae60', width=2),
        marker=dict(size=6)
    ),
    row=3, col=1, secondary_y=True
)

# Update axes labels
fig_trends.update_yaxes(title_text="Fraud Detection Metrics (%)", row=1, col=1)
fig_trends.update_yaxes(title_text="Alert Processing Metrics (%)", row=2, col=1)
fig_trends.update_yaxes(title_text="Fraud Value ($)", row=3, col=1, secondary_y=False)
fig_trends.update_yaxes(title_text="Prevented Value ($)", row=3, col=1, secondary_y=True)
fig_trends.update_xaxes(title_text="Time Period", row=3, col=1)

fig_trends.update_layout(
    title_text=f"<b>Executive Dashboard - Trend Analysis</b><br><sub>Time Period: {time_granularity.upper()} | Granularity: {time_granularity}</sub>",
    height=1000,
    width=1400,
    hovermode='x unified',
    showlegend=True,
    legend=dict(x=0.01, y=0.99)
)

print("✓ Time-series trend charts created (3 subplots with 10 metrics)")
print("  Metrics tracked: Fraud Detection, Alert Processing, Financial Impact")

# Display the figure
fig_trends.show()



VISUALIZATION: TIME-SERIES TRENDS


✓ Time-series trend charts created (3 subplots with 10 metrics)
  Metrics tracked: Fraud Detection, Alert Processing, Financial Impact


In [16]:
# ============================================================================
# SECTION 8.3: RENDER SUMMARY TABLE (Executive Summary)
# ============================================================================

print("\n" + "="*80)
print("VISUALIZATION: EXECUTIVE SUMMARY TABLE")
print("="*80)

# Create comprehensive summary table
summary_data = {
    'KPI': [
        'Fraud Rate (%)',
        'Precision (%)',
        'True Positive Rate (%)',
        'False Positive Rate (%)',
        'Evaluated Alert Rate (%)',
        'Interdiction Rate (%)',
        'MTTD (Days)',
        'Total Fraud Value ($)',
        'Prevented Fraud Value ($)',
        'Average Fraud Value ($)'
    ],
    'Current Value': [
        f"{current_fraud_rate:.2f}%",
        f"{current_precision:.2f}%",
        f"{current_tpr:.2f}%",
        f"{current_fpr:.2f}%",
        f"{current_evaluated_alert_rate:.2f}%",
        f"{current_interdiction_rate:.2f}%",
        f"{current_mttd:.2f}",
        f"${current_total_fraud_value:,.2f}",
        f"${current_prevented_value:,.2f}",
        f"${current_avg_fraud_value:,.2f}"
    ],
    'Previous Period': [
        f"{fraud_perf_pd['fraud_rate_pct'].iloc[-2]:.2f}%" if len(fraud_perf_pd) >= 2 else "N/A",
        f"{fraud_perf_pd['precision_calc'].iloc[-2]:.2f}%" if len(fraud_perf_pd) >= 2 else "N/A",
        f"{fraud_perf_pd['tpr_calc'].iloc[-2]:.2f}%" if len(fraud_perf_pd) >= 2 else "N/A",
        f"{case_perf_pd['false_positive_rate_pct'].iloc[-2]:.2f}%" if len(case_perf_pd) >= 2 else "N/A",
        f"{alert_perf_pd['evaluated_alert_rate_calc'].iloc[-2]:.2f}%" if len(alert_perf_pd) >= 2 else "N/A",
        f"{alert_perf_pd['overall_interdiction_rate_pct'].iloc[-2]:.2f}%" if len(alert_perf_pd) >= 2 else "N/A",
        f"{case_perf_pd['mttd_days'].iloc[-2]:.2f}" if len(case_perf_pd) >= 2 else "N/A",
        f"${financial_pd['total_fraud_value'].iloc[-2]:,.2f}" if len(financial_pd) >= 2 else "N/A",
        f"${financial_pd['prevented_fraud_value_calc'].iloc[-2]:,.2f}" if len(financial_pd) >= 2 else "N/A",
        f"${financial_pd['average_fraud_value'].iloc[-2]:,.2f}" if len(financial_pd) >= 2 else "N/A"
    ],
    'Trend': [
        trends_df['fraud_rate_direction'].iloc[-1] if len(trends_df) > 0 and 'fraud_rate_direction' in trends_df.columns else "→",
        trends_df['precision_direction'].iloc[-1] if len(trends_df) > 0 and 'precision_direction' in trends_df.columns else "→",
        trends_df['tpr_direction'].iloc[-1] if len(trends_df) > 0 and 'tpr_direction' in trends_df.columns else "→",
        trends_df['fpr_direction'].iloc[-1] if len(trends_df) > 0 and 'fpr_direction' in trends_df.columns else "→",
        trends_df['idr_direction'].iloc[-1] if len(trends_df) > 0 and 'idr_direction' in trends_df.columns else "→",
        trends_df['idr_direction'].iloc[-1] if len(trends_df) > 0 and 'idr_direction' in trends_df.columns else "→",
        "→",
        trends_df['fraud_value_direction'].iloc[-1] if len(trends_df) > 0 and 'fraud_value_direction' in trends_df.columns else "→",
        trends_df['fraud_value_direction'].iloc[-1] if len(trends_df) > 0 and 'fraud_value_direction' in trends_df.columns else "→",
        trends_df['fraud_value_direction'].iloc[-1] if len(trends_df) > 0 and 'fraud_value_direction' in trends_df.columns else "→",
    ],
    '% Change': [
        f"{trends_df['fraud_rate_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['precision_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['tpr_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['fpr_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['ear_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['idr_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['mttd_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['fraud_value_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['fraud_value_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
        f"{trends_df['fraud_value_change_pct'].iloc[-1]:+.1f}%" if len(trends_df) > 0 else "N/A",
    ]
}

summary_table_df = pd.DataFrame(summary_data)

# Create Plotly table
fig_summary = go.Figure(data=[go.Table(
    header=dict(
        values=list(summary_table_df.columns),
        fill_color='#2c3e50',
        align='left',
        font=dict(color='white', size=12, family="Arial")
    ),
    cells=dict(
        values=[summary_table_df[col] for col in summary_table_df.columns],
        fill_color='#ecf0f1',
        align='left',
        font=dict(size=11, family="Arial"),
        height=30
    )
)])

fig_summary.update_layout(
    title=f"<b>Executive Summary Table</b><br><sub>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')} | Granularity: {time_granularity.upper()}</sub>",
    height=600,
    width=1400,
    margin=dict(l=20, r=20, t=100, b=20)
)

print("✓ Executive summary table created with 10 KPIs")
print("  Columns: Current Value, Previous Period, Trend Direction, % Change")

# Display the figure
fig_summary.show()

print("\n" + "="*80)
print("✓ ALL VISUALIZATIONS COMPLETED SUCCESSFULLY")
print("="*80)



VISUALIZATION: EXECUTIVE SUMMARY TABLE
✓ Executive summary table created with 10 KPIs
  Columns: Current Value, Previous Period, Trend Direction, % Change



✓ ALL VISUALIZATIONS COMPLETED SUCCESSFULLY


In [17]:
# ============================================================================
# SECTION 9: EXPORT RESULTS (JSON, CSV, PDF formats)
# ============================================================================

print("\n" + "="*80)
print("EXPORT RESULTS")
print("="*80)

import json
from datetime import datetime

export_timestamp = pd.Timestamp.now()
export_dir = f"{WAREHOUSE_ROOT}/exports"

# Create export directory if it doesn't exist
import os
os.makedirs(export_dir, exist_ok=True)

# ===== EXPORT 1: JSON METADATA =====
metadata = {
    'dashboard_name': 'Executive Overview Dashboard',
    'generated_timestamp': export_timestamp.isoformat(),
    'date_range': {
        'start_date': dashboard_start_date.strftime('%Y-%m-%d'),
        'end_date': dashboard_end_date.strftime('%Y-%m-%d'),
        'range_type': date_range_type
    },
    'time_granularity': time_granularity,
    'metrics_summary': {
        'fraud_rate_pct': float(current_fraud_rate),
        'precision_pct': float(current_precision),
        'tpr_pct': float(current_tpr),
        'fpr_pct': float(current_fpr),
        'evaluated_alert_rate_pct': float(current_evaluated_alert_rate),
        'interdiction_rate_pct': float(current_interdiction_rate),
        'mttd_days': float(current_mttd),
        'total_fraud_value': float(current_total_fraud_value),
        'prevented_fraud_value': float(current_prevented_value),
        'average_fraud_value': float(current_avg_fraud_value)
    },
    'data_sources': [
        'tms_metrics_gold',
        'fraud_metrics_gold',
        'alert_metrics_gold',
        'case_metrics_gold'
    ]
}

metadata_file = f"{export_dir}/dashboard_metadata_{export_timestamp.strftime('%Y%m%d_%H%M%S')}.json"
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ JSON Metadata exported: {metadata_file}")

# ===== EXPORT 2: CSV - Trend Data =====
csv_file = f"{export_dir}/dashboard_trends_{export_timestamp.strftime('%Y%m%d_%H%M%S')}.csv"
trends_export = trends_df[['period', 'fraud_rate', 'precision', 'tpr', 'fpr',
                           'evaluated_alert_rate', 'interdiction_rate', 'mttd',
                           'total_fraud_value', 'fraud_rate_change_pct', 'precision_change_pct']].copy()
trends_export.to_csv(csv_file, index=False)

print(f"✓ CSV Trends exported: {csv_file}")

# ===== EXPORT 3: CSV - Summary Metrics =====
summary_csv_file = f"{export_dir}/dashboard_summary_{export_timestamp.strftime('%Y%m%d_%H%M%S')}.csv"
summary_table_df.to_csv(summary_csv_file, index=False)

print(f"✓ CSV Summary exported: {summary_csv_file}")

# ===== EXPORT 4: HTML Dashboard (for distribution) =====
html_file = f"{export_dir}/dashboard_{export_timestamp.strftime('%Y%m%d_%H%M%S')}.html"

# Combine all visualizations into single HTML
with open(html_file, 'w') as f:
    f.write(f"""
    <html>
        <head>
            <title>Executive Overview Dashboard</title>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
                h1 {{ color: #2c3e50; text-align: center; }}
                .metadata {{ background-color: white; padding: 15px; margin: 10px 0; border-radius: 5px; }}
                .chart-container {{ background-color: white; padding: 20px; margin: 20px 0; border-radius: 5px; }}
                .export-info {{ color: #555; font-size: 12px; margin-top: 30px; }}
            </style>
        </head>
        <body>
            <h1>Executive Overview Dashboard</h1>
            
            <div class="metadata">
                <h2>Dashboard Metadata</h2>
                <p><strong>Generated:</strong> {export_timestamp.strftime('%Y-%m-%d %H:%M:%S')}</p>
                <p><strong>Date Range:</strong> {dashboard_start_date.strftime('%Y-%m-%d')} to {dashboard_end_date.strftime('%Y-%m-%d')} ({date_range_type})</p>
                <p><strong>Time Granularity:</strong> {time_granularity.upper()}</p>
                <p><strong>Data Source:</strong> Gold-Layer Hudi Tables</p>
            </div>
            
            <div class="chart-container">
                <h2>Current KPI Values</h2>
                {fig_kpis.to_html(include_plotlyjs=False, div_id="kpi_chart")}
            </div>
            
            <div class="chart-container">
                <h2>Trend Analysis</h2>
                {fig_trends.to_html(include_plotlyjs=False, div_id="trends_chart")}
            </div>
            
            <div class="chart-container">
                <h2>Executive Summary</h2>
                {fig_summary.to_html(include_plotlyjs=False, div_id="summary_chart")}
            </div>
            
            <div class="export-info">
                <p>This dashboard was automatically generated and contains pre-aggregated metrics from the Fraud Detection System.</p>
                <p>For detailed analysis, refer to the detailed metrics dashboard.</p>
            </div>
        </body>
    </html>
    """)

print(f"✓ HTML Dashboard exported: {html_file}")

print("\n" + "="*80)
print("✓ ALL EXPORTS COMPLETED")
print("="*80)
print(f"\nExport Summary:")
print(f"  - JSON Metadata: {metadata_file}")
print(f"  - CSV Trends: {csv_file}")
print(f"  - CSV Summary: {summary_csv_file}")
print(f"  - HTML Dashboard: {html_file}")
print(f"\nNote: PDF export requires nbconvert. Run in terminal:")
print(f"  jupyter nbconvert --to pdf db_us2.ipynb --output dashboard_{export_timestamp.strftime('%Y%m%d_%H%M%S')}.pdf")



EXPORT RESULTS
✓ JSON Metadata exported: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_metadata_20260417_063358.json
✓ CSV Trends exported: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_trends_20260417_063358.csv
✓ CSV Summary exported: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_summary_20260417_063358.csv
✓ HTML Dashboard exported: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_20260417_063358.html

✓ ALL EXPORTS COMPLETED

Export Summary:
  - JSON Metadata: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_metadata_20260417_063358.json
  - CSV Trends: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_trends_20260417_063358.csv
  - CSV Summary: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_summary_20260417_063358.csv
  - HTML Dashboard: /home/muneeb-faraz/Hudi_Test_Postgres/exports/dashboard_20260417_063358.html

Note: PDF export requires nbconvert. Run in terminal:
  jupyter nbconvert --to pdf db_us2.ipynb --output dash